In [13]:
import pandas as pd
from tqdm.auto import tqdm
import json
import os
import shutil

In [10]:
df_cluster = pd.read_csv("/home/aethercore/Doutoramento/PepBench_dataset/Dataset_curation/clustering/clu_cluster.tsv", sep='\t', names=["rep", "member"])
df_cluster["rep"] = df_cluster["rep"].apply(lambda x: x.lower())
df_cluster["member"] = df_cluster["member"].apply(lambda x: x.lower())
df_cluster["rep_id"] = df_cluster["rep"].apply(lambda x: x.split("_")[0].lower())
df_cluster["member_id"] = df_cluster["member"].apply(lambda x: x.split("_")[0].lower())
cluster_map = df_cluster.groupby('rep_id')['member_id'].apply(set).to_dict()

df_query = pd.read_csv("/home/aethercore/PepPCBench/job_list_updated.csv")
df_query_set = set(df_query["pdb_id"])
print(df_query_set)


core_set = set()
train_set = set()

for pdb_id in tqdm(df_query_set):
    if pdb_id in df_cluster['rep_id'].values:
        filter_df = df_cluster[df_cluster["rep_id"] == pdb_id] # This gives me the composition of the cluste
        cluster_ids = filter_df["member_id"].unique()
        if all(member_id in df_query_set for member_id in cluster_ids):
            core_set.add(pdb_id)
        else:
            train_set.add(pdb_id)

print(f"core_set: {len(core_set)}, train_set_non_redundant: {len(train_set)}")

print(train_set)
print(core_set)

{'8t32', '8isn', '8q3s', '8rmo', '8t5p', '8tuq', '8u26', '8tq9', '8r1t', '8wxu', '8jis', '8sbk', '8tdx', '8q7k', '8u2y', '9cwn', '8u1z', '8q6d', '8vjy', '8ttu', '8cmd', '8q6e', '8vgc', '8i5e', '8pxx', '8q5p', '8vmf', '8tg7', '9f7x', '8gjf', '8wo9', '8s6n', '8jgf', '8rlt', '8jgg', '8wxt', '8g8q', '8p9w', '8q7i', '8wxx', '8t8s', '8pwm', '8ib1', '9ck0', '8s8o', '8wef', '9ez1', '8cmi', '8vjx', '8vsj', '8pz7', '8jqt', '8x8a', '8pwf', '8fyu', '8sm5', '8rxb', '8q1q', '8tx8', '8rhq', '8jbh', '8pku', '8pwe', '8sz2', '9ax9', '8w6l', '8p9r', '8ym2', '8qlk', '8pef', '8wul', '8tq7', '8pii', '8pn5', '8jly', '8gjg', '8rbu', '8ryn', '8tos', '8u2m', '8sud', '8wxv', '9gag', '9eyr', '8sgf', '8j5u', '8ui6', '8jzd', '8vcx', '8vjp', '8wxz', '8t8g', '8oio', '8pz8', '8vy8', '8rng', '8sr6', '8w6a', '8rlv', '9g13', '8pjg', '8pkv', '8cd3', '8ttt', '8r1n', '8tuv', '8kb1', '8jzw', '8t2n', '8tee', '8q3t', '8fud', '8q26', '8kb0', '8w5z', '8rcv', '8th1', '8p9x', '8jgb', '8zpt', '8jyq', '8tbw', '8tor', '8yhz', '8cmg',

100%|██████████| 240/240 [00:00<00:00, 640.36it/s]

core_set: 135, train_set_non_redundant: 6
{'8r10', '8rbv', '8rym', '8wxz', '8tor', '8qlk'}
{'8t5p', '8q3s', '8tq9', '8r1t', '8wxu', '8sbk', '8tdx', '9cwn', '8q6d', '8vjy', '8ttu', '8cmd', '8q6e', '8vgc', '8vmf', '8q5p', '9f7x', '8tg7', '8gjf', '8wo9', '8s6n', '8jgg', '8g8q', '8q7i', '8wxx', '8t8s', '8pwm', '8ib1', '9ez1', '8wef', '8cmi', '8vsj', '8pz7', '8jqt', '8x8a', '8rxb', '8q1q', '8jbh', '8pku', '8pwe', '8sz2', '9ax9', '8w6l', '8p9r', '8ym2', '8pef', '8wul', '8tq7', '8pii', '8pn5', '8jly', '8gjg', '8sud', '8wxv', '8j5u', '8jzd', '8vjp', '8vy8', '8sr6', '8w6a', '9g13', '8cd3', '8ttt', '8r1n', '8jzw', '8tee', '8q3t', '8fud', '8kb0', '8p9x', '8jgb', '8jyq', '8yhz', '8cmg', '8ofg', '8iya', '8g8c', '8j4g', '8jj9', '8vdd', '8wee', '8p0q', '8smo', '8wg8', '8igc', '8t5q', '8u77', '8r1p', '8ral', '8qlg', '8pz6', '8pzb', '8v8e', '8jjv', '8vcy', '8gji', '8tfu', '8okf', '8tmz', '8kcv', '8q2z', '8p6i', '8og0', '8suv', '8u9g', '8g8a', '8qxw', '8th5', '8tbv', '8tg8', '8vd2', '8tqa', '8vju', '8c5

In [11]:
with open("/home/aethercore/Doutoramento/PepBench_dataset/Dataset_curation/clustering/clusters_query.json", "w") as f:
    json.dump(
        {
            "core_set": list(core_set),
        }, f, indent=4
    )

In [14]:
for pdb_id in list(core_set):
    for structures in os.listdir(f"/home/aethercore/Doutoramento/PepBench_dataset/Dataset_curation/pdb_test_data/Static"):
        if f'{pdb_id}.cif' in structures:
            shutil.copy2(f"/home/aethercore/Doutoramento/PepBench_dataset/Dataset_curation/pdb_test_data/Static/{structures}", f"/home/aethercore/Doutoramento/PepBench_dataset/benchmark_sets/assemblies/core_set")
            break